# 21_agentcore: executable lesson notebook

Read [the guide](README.md). These cells mirror every script and run offline by default.

In [ ]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists() and (p / "data").is_dir()
)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"
os.environ["MEM0_TELEMETRY"] = "false"
from common.network import offline_network


def run_lesson(entry_point, script):
    """Replay the script offline, on a fresh worker event loop when needed."""

    def work():
        previous = sys.argv
        sys.argv = [script, "--mock"]
        try:
            with offline_network():
                entry_point()
        finally:
            sys.argv = previous

    with ThreadPoolExecutor(max_workers=1) as pool:
        pool.submit(work).result(timeout=120)

## 01_memory

Source: `modules/21_agentcore/examples/01_memory.py`.

In [ ]:
# Source: modules/21_agentcore/examples/01_memory.py
"""Short-term events and long-term preferences: Nimbus production lesson.

Run: uv run --all-groups python modules/21_agentcore/examples/01_memory.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

from common.agentcore_memory import preferences
from common.config import course_value
from common.lessons import complete
from common.pretty import run_cli
from common.production_lessons import aws_settings


def main() -> None:
    _, cfg = aws_settings()
    memory = (
        "NimbusDemo-1234567890" if cfg.provider == "mock" else course_value("AGENTCORE_MEMORY_ID")
    )
    # KEY IDEA: this standalone synthetic demo binds CUS-001 in trusted host code.
    print(preferences(cfg, memory, "CUS-001"))
    complete()


run_lesson(main, "modules/21_agentcore/examples/01_memory.py")

## 02_gateway

Source: `modules/21_agentcore/examples/02_gateway.py`.

In [ ]:
# Source: modules/21_agentcore/examples/02_gateway.py
"""Gateway discovery, search and input policy: Nimbus production lesson.

Run: uv run --all-groups python modules/21_agentcore/examples/02_gateway.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

from common.config import course_value
from common.gateway_booking import gateway_probe
from common.gateway_openapi import openapi_target
from common.lessons import complete
from common.pretty import run_cli
from common.production_lessons import aws_settings


def main() -> None:
    _, cfg = aws_settings()
    request = openapi_target(
        cfg,
        "nimbus-booking-abcdefghij",
        f"https://abc123.execute-api.{cfg.aws_region}.amazonaws.com/demo",
    )
    print("OpenAPI request shape validated locally:", request["name"])
    print("Schema fixture only; registering a target needs a deployed IAM-protected API.")
    # KEY IDEA: the server enforces the amount limit even if the model asks for more.
    print(
        gateway_probe(
            cfg,
            url=course_value("AGENTCORE_GATEWAY_URL"),
            token=course_value("NIMBUS_ACCESS_TOKEN"),
        )
    )
    complete()


run_lesson(main, "modules/21_agentcore/examples/02_gateway.py")

## 03_identity

Source: `modules/21_agentcore/examples/03_identity.py`.

In [ ]:
# Source: modules/21_agentcore/examples/03_identity.py
"""Inbound JWT verification and outbound OAuth: Nimbus production lesson.

Run: uv run --all-groups python modules/21_agentcore/examples/03_identity.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

from common.agentcore_identity import cognito_identity, demo_identity, outbound_oauth
from common.config import course_value
from common.lessons import complete
from common.pretty import run_cli
from common.production_lessons import aws_settings


def main() -> None:
    _, cfg = aws_settings()
    if cfg.provider == "mock":
        print(demo_identity())
        token, workload, provider = "offline-token", "NimbusCopilot", "partner"
    else:
        token = course_value("NIMBUS_ACCESS_TOKEN")
        print(
            cognito_identity(
                cfg, token, course_value("NIMBUS_USER_POOL_ID"), course_value("NIMBUS_CLIENT_ID")
            )
        )
        workload, provider = "NimbusCopilot", course_value("NIMBUS_OAUTH_PROVIDER")
    # KEY IDEA: third-party tokens never enter the agent's context or output.
    print(outbound_oauth(cfg, workload=workload, provider=provider, user_token=token))
    complete()


run_lesson(main, "modules/21_agentcore/examples/03_identity.py")

## 04_code_interpreter

Source: `modules/21_agentcore/examples/04_code_interpreter.py`.

In [ ]:
# Source: modules/21_agentcore/examples/04_code_interpreter.py
"""Code Interpreter with fixed reviewed Python: Nimbus production lesson.

Run: uv run --all-groups python modules/21_agentcore/examples/04_code_interpreter.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

from common.agentcore_tools import builtin_agent
from common.lessons import complete
from common.pretty import run_cli
from common.production_lessons import aws_settings


def main() -> None:
    _, cfg = aws_settings()
    # KEY IDEA: executeCode is the confirmed API method; stop in finally on every outcome.
    print(builtin_agent(cfg, "code"))
    complete()


run_lesson(main, "modules/21_agentcore/examples/04_code_interpreter.py")

## 05_browser

Source: `modules/21_agentcore/examples/05_browser.py`.

In [ ]:
# Source: modules/21_agentcore/examples/05_browser.py
"""Browser session reading a synthetic receipt: Nimbus production lesson.

Run: uv run --all-groups python modules/21_agentcore/examples/05_browser.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

from common.agentcore_tools import builtin_agent
from common.lessons import complete
from common.pretty import run_cli
from common.production_lessons import aws_settings


def main() -> None:
    _, cfg = aws_settings()
    # KEY IDEA: host-owned HTML avoids arbitrary sites, logins and model-selected URLs.
    print(builtin_agent(cfg, "browser"))
    complete()


run_lesson(main, "modules/21_agentcore/examples/05_browser.py")

## 06_observability

Source: `modules/21_agentcore/examples/06_observability.py`.

In [ ]:
# Source: modules/21_agentcore/examples/06_observability.py
"""Bounded service telemetry: Nimbus production lesson.

Run: uv run --all-groups python modules/21_agentcore/examples/06_observability.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

import asyncio

from common.lessons import complete
from common.pretty import run_cli
from common.production_checks import emit_metrics
from common.production_lessons import aws_settings
from common.production_support import answer_support


def main() -> None:
    _, cfg = aws_settings()
    result = asyncio.run(answer_support({"prompt": "Refund time?"}, cfg))
    # KEY IDEA: CloudWatch receives numeric usage/latency, never the prompt or answer.
    print(emit_metrics(cfg, result))
    complete()


run_lesson(main, "modules/21_agentcore/examples/06_observability.py")

## 07_policy

Source: `modules/21_agentcore/examples/07_policy.py`.

In [ ]:
# Source: modules/21_agentcore/examples/07_policy.py
"""Refund policy boundary: Nimbus production lesson.

Run: uv run --all-groups python modules/21_agentcore/examples/07_policy.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

from common.config import course_value
from common.gateway_booking import gateway_probe
from common.lessons import complete
from common.pretty import run_cli
from common.production_lessons import aws_settings


def main() -> None:
    _, cfg = aws_settings()
    # KEY IDEA: the server enforces the amount limit even if the model asks for more.
    print(
        gateway_probe(
            cfg,
            url=course_value("AGENTCORE_GATEWAY_URL"),
            token=course_value("NIMBUS_ACCESS_TOKEN"),
        )
    )
    complete()


run_lesson(main, "modules/21_agentcore/examples/07_policy.py")

## 08_evaluations

Source: `modules/21_agentcore/examples/08_evaluations.py`.

In [ ]:
# Source: modules/21_agentcore/examples/08_evaluations.py
"""Managed evaluations with explicit scheduling: Nimbus production lesson.

Run: uv run --all-groups python modules/21_agentcore/examples/08_evaluations.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

import argparse
from pathlib import Path

from common.lessons import complete
from common.managed_evaluation import evaluate_spans
from common.pretty import run_cli
from common.production_lessons import aws_settings


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--spans", type=Path, help="Reviewed CloudWatch export for one synthetic session"
    )
    args, cfg = aws_settings(parser)
    # KEY IDEA: metadata-only traces cannot establish answer helpfulness.
    print(evaluate_spans(cfg, args.spans))
    complete()


run_lesson(main, "modules/21_agentcore/examples/08_evaluations.py")

## 09_integrated_copilot

Source: `modules/21_agentcore/examples/09_integrated_copilot.py`.

In [ ]:
# Source: modules/21_agentcore/examples/09_integrated_copilot.py
"""Identity, Memory and Gateway integration: Nimbus production lesson.

Run: uv run --all-groups python modules/21_agentcore/examples/09_integrated_copilot.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

import json

from common.lessons import complete
from common.pretty import run_cli
from common.production_copilot import copilot
from common.production_lessons import aws_settings


def main() -> None:
    _, cfg = aws_settings()
    # KEY IDEA: verified identity binds Memory's actor outside model arguments.
    print(json.dumps(copilot(cfg), indent=2))
    complete()


run_lesson(main, "modules/21_agentcore/examples/09_integrated_copilot.py")

## production_copilot

Source: `modules/21_agentcore/use_case/production_copilot.py`.

In [ ]:
# Source: modules/21_agentcore/use_case/production_copilot.py
"""Nimbus production Copilot: Nimbus production lesson.

Run: uv run --all-groups python modules/21_agentcore/use_case/production_copilot.py --mock
Expected: Lesson complete. Mock cost/resources: $0 / none.
Live AWS calls require --live-aws where applicable; see module cost and cleanup tables.
"""

import json

from common.lessons import complete
from common.pretty import run_cli
from common.production_copilot import copilot
from common.production_lessons import aws_settings


def main() -> None:
    _, cfg = aws_settings()
    # KEY IDEA: verified identity binds Memory's actor outside model arguments.
    print(json.dumps(copilot(cfg), indent=2))
    complete()


run_lesson(main, "modules/21_agentcore/use_case/production_copilot.py")